## Co trzeba sprawdzić

- Jak zmienia się masa mięsa w muszli?  
  → Czy istnieje limit jej wzrostu?

- Jak dokładnie mierzona jest masa mięsa (`weight of meat`)?

- czy zmienne są jakoś standaryzowane

- Błąd pomiarowy – jak go oszacować albo sprawdzić?

- Jakie są używane metody pomiarowe?

## propozycje

- takie tempo wzrostu na początku znacznie szybciej ich przybywa w stosunku do naszej zmiennej a potem spowalnia (organizm nie rośnie cały czas) 



In [ ]:
# basic 
import pandas as pd
import numpy as np
import scipy.stats as stats

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# statistics
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from statsmodels.tools.tools import add_constant

# models 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
# from sklearn.metrics import 

# data
from ucimlrepo import fetch_ucirepo 

# global parameters
seed = 1333

helper function

In [ ]:
# vifs calculation function
def get_vifs(df):
    '''
    df need to be dataframe only with parameters to compare and with column names for print
    '''
    col_names = df.columns
    df = add_constant(df)
    vif_vals = [vif(df, i) for i in range(1, df.shape[1])]
    pd_ser_rtn = pd.Series(vif_vals, index=col_names)
    return pd_ser_rtn



def F_importance_calculation(df, y, test_cols):
    '''
    Function takes DataFrame (df) with all features (implemented & new) and tests if new features (test_cols), bring significant difference.
    '''
    # creating basic feature df & model 
    X = df.drop(columns=test_cols)
    # X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.2, random_state=seed)
    model_1 = sm.OLS(y, add_constant(X)).fit()
    # create table with all results (adding context WITH FEATURE NAME)
    compare_table = None

    # TODO create loop (just range)
    for col_name in test_cols:
        # new features list
        X_new = pd.concat([X, df[col_name]], axis=1)
        # creating models (sm)
        model_2 = sm.OLS(y, add_constant(X_new)).fit()
        # calculating values for every model change
        new_results = sm.stats.anova_lm(model_1, model_2)
        new_results['Model'] = pd.Series(['Model Standardowy', col_name]) 
        new_results = new_results[list(new_results.columns[-1:]) + list(new_results.columns[:-1])]
        # TODO UPDATE table
        if compare_table is None:
            # create template
            compare_table = new_results
        else:
            # update data 
            compare_table = pd.concat([compare_table, new_results.iloc[[1], :]], axis=0)
    # returning df with data 
    return compare_table.set_index('Model')

load data from server 

In [ ]:
# fetch dataset 
abalone = fetch_ucirepo(id=1) 

# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets 
df = pd.concat([X, y], axis=1)

# saving data 
# df.to_csv('abalone.csv', index=False)
# df.to_excel('abalone.xlsx')

Load data from file

In [ ]:
# parameters 
col_names = ['Płeć', 'Długość', 'Średnica', 'Wysokość', 'Cała_waga', 'Waga_po_obraniu', 'Waga_trzewi', 'Waga_powłoki', 'Pierścienie']

# loading data
df = pd.read_csv('abalone.csv', names=col_names, header=0)
X = df.drop('Pierścienie', axis=1)
y = df[['Pierścienie']]

# decoding categorical data (as boolean) 
X['Płeć_M'] = (X['Płeć'] == 'M')
X['Płeć_F'] = (X['Płeć'] == 'F')
X['Płeć_I'] = (X['Płeć'] == 'I')
X = X[list(X.columns[-3:]) + list(X.columns[1:-3])]

X


analiza korelacji 

In [ ]:
# -------- parameters ------- #
sex_columns = ['Płeć_I', 'Płeć_M', 'Płeć_F']

# ------- correlation ------- #
fig, ax = plt.subplots(1, 2, figsize=(20, 8))

# sns.heatmap((X[X['Płeć_I'] == True].iloc[:, 3:]).corr(), 
#             annot=True, cmap='coolwarm', vmin=0, vmax=1,
#             ax=ax[0])
# ax[0].set_title('Korelacja dla I')

sns.heatmap(X[X['Płeć_M'] == True].drop(columns=sex_columns).corr(), 
            annot=True, cmap='coolwarm', vmin=0, vmax=1,
            ax=ax[0])
ax[0].set_title('Korelacja dla M')

sns.heatmap(X[X['Płeć_F'] == True].drop(columns=sex_columns).corr(), 
            annot=True, cmap='coolwarm', vmin=0, vmax=1,
            ax=ax[1])
ax[1].set_title('Korelacja dla F')

# ------- vif ------- #
# standard 
print(f"vif'y dla standardowych wartości\n")
print(get_vifs(X.drop(columns=sex_columns)))
# reduced features 
reduce_features = ['Średnica', 'Cała_waga']
print(f"\n\nVif'y po redukcji parametrów\n")
print(get_vifs(X.drop(columns=sex_columns + reduce_features)))


próbne modele regresji 

In [ ]:
# taking into consideration only model with one sex (later extend for different) 

# basic 
filt = X['Płeć_F'] == True
y_local = y[filt]
X_old = X[filt].drop(columns=sex_columns+reduce_features).copy()
# new features 
X_new = X_old.copy()
X_new['Masa/objętość'] = X['Cała_waga'] * X['Długość'] * X['Średnica'] * X['Wysokość']
X_new['Waga_powłoki/Waga_po_obraniu'] = X['Waga_powłoki'] / X['Waga_po_obraniu']
X_new['Waga_powłoki/Waga_trzewi'] = X['Waga_powłoki'] / X['Waga_trzewi']
X_new['Waga_powłoki/Cała_waga'] = X['Waga_powłoki'] / X['Cała_waga']
X_new['Długość/Wysokość'] = X['Długość'] / X['Średnica'] 

# finding results
results = F_importance_calculation(X_new, y_local, list(X_new.columns[X_old.shape[1]:]))
results['Odrzucenie H_0'] = results['Pr(>F)'] < 5e-2
drop_features = list(results[results['Pr(>F)'] >= 5e-2].index)
# print 
results

In [ ]:
# taking into consideration only model with one sex (later extend for different) 

# basic 
filt = X['Płeć_F'] == True
y_local = y[filt]
X_old = X[filt].drop(columns=sex_columns+reduce_features).copy()
# new features 
X_new = X_old.copy()
# stosunki wymiarów 
X_new['Długość/Wysokość'] = X['Długość'] / X['Średnica']
# masa / objętość 
X_new['Masa/objętość'] = X['Cała_waga'] * X['Długość'] * X['Średnica'] * X['Wysokość']
# stosunki mas 
X_new['Waga_powłoki/Waga_po_obraniu'] = X['Waga_powłoki'] / X['Waga_po_obraniu']
X_new['Waga_powłoki/Waga_trzewi'] = X['Waga_powłoki'] / X['Waga_trzewi']
X_new['Waga_powłoki/Cała_waga'] = X['Waga_powłoki'] / X['Cała_waga']
X_new['Waga_po_obraniu/Cała_waga'] = X['Waga_po_obraniu'] / X['Cała_waga']
X_new['Waga_trzewi/Cała_waga'] = X['Waga_trzewi'] / X['Cała_waga']
X_new['Waga_powłoki/Cała_waga'] = X['Waga_powłoki'] / X['Cała_waga']

# finding results
results = F_importance_calculation(X_new, y_local, list(X_new.columns[X_old.shape[1]:]))
results['Odrzucenie H_0'] = results['Pr(>F)'] < 5e-2
drop_features = list(results[results['Pr(>F)'] >= 5e-2].index)
# print 
results

Na poziomie istotności przyjmiemy dodatkowe zmienne:
- Waga_powłoki/Waga_po_obraniu
- Waga_powłoki/Waga_trzewi
- Długość/Wysokość 

In [ ]:
# checking new vifs
X_new_features = X_new.drop(columns=drop_features)
print(get_vifs(X_new_features))
X_new_features_low_vif = X_new_features.drop(columns=['Waga_po_obraniu', 'Waga_trzewi', 'Waga_powłoki/Waga_po_obraniu'])
# te kolumny były sprawdzone wcześniej
print(get_vifs(X_new_features_low_vif))
# features df for models 
f1 = X_new
f2 = X_new_features_low_vif

lm1, lm2 = LinearRegression(), LinearRegression()
lm1.fit(f1, y_local), lm2.fit(f2, y_local)
scr1, scr2 = lm1.score(f1, y_local), lm2.score(f2, y_local)
scr1, scr2 # porównanie pomiędzy modelem ze wszystkimi wartościami (po redukcji pierwszych vif'ów) vs zredukowany



Naszym zadaniem teraz QQ plotu (jakiego rodzaju jest błąd)

In [ ]:
residuals = y_local - lm2.predict(f2)


plt.scatter(lm2.predict(f2), residuals)
plt.show()

residuals = residuals.sort_values('Pierścienie')
sm.qqplot(residuals, line='s')
plt.title('Wykres QQ reszt modelu')
plt.show()
# bez sensu ten model



# testy
result1 = stats.shapiro(residuals)

print(result1)


Propozycje jeżeli chodzi o model:

- będzie trzeba rozbić model na 3 płcie, jeden oparty do tych wczesnych pierścieni, do określonego wieku (trzeba na danych zobaczyć), reszte trzeba 

- skorzystamy ze zmiennych które dostajemy poprzez 

- mogą nas interesowac wagi ale to trzeba sprawdzić po kolei 

- będzie nas interesować to że wiek nie jest liniowy ze wzrostem, najczęściej wzrost jest bardzo dużo w przypadku młodych osobników, mały w przypadku dużych 

- błąd jest bardzo dziwny, zatem stosowanie testów nie jest uzasadnione ~ jak to zmienic bo to mocno utrudnia wytłumaczenie tych wniosków, trzeba zupełnie inne testy zastosować (dla nie dla rozkładów normalnych )